# Incertitude par distance de features

Ce notebook teste une méthode plus ciblée sur la limite observée dans FreshOrRotten.

Le `unseen_category_split` montre que le CNN généralise moins bien à des produits absents de l'entraînement. L'idée est donc de vérifier si ces images sont éloignées des représentations apprises sur le `training_set`.

## Principe

On utilise le CNN comme extracteur de features :

```text
image -> CNN -> représentation interne -> distance aux prototypes du train
```

Pour chaque classe prédite, on calcule la distance entre l'image et le prototype de cette classe dans l'espace de features.

Si la distance est trop grande, la prédiction devient `uncertain`.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

reports_dir = project_root / "reports"
project_root

## Lancer l'analyse

Cette cellule utilise les modèles et les splits déjà sauvegardés.
Elle peut prendre quelques minutes car elle extrait les features de train, validation et test.

In [ ]:
command = [sys.executable, str(project_root / "src" / "feature_distance_uncertainty.py"), "--protocol", "both"]
subprocess.run(command, cwd=project_root, check=True)

## Résultats globaux

In [ ]:
feature_metrics = pd.read_csv(reports_dir / "feature_distance_metrics.csv")
feature_metrics

In [ ]:
plot_columns = ["baseline_accuracy", "accepted_accuracy", "coverage", "uncertain_rate"]
ax = feature_metrics.set_index("protocol")[plot_columns].plot(kind="bar", figsize=(9, 5))
ax.set_ylim(0, 1)
ax.set_title("Incertitude par distance de features")
ax.set_ylabel("Valeur")
ax.legend(loc="lower right")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Comparaison avec l'incertitude sigmoid

Si `uncertainty_metrics.csv` existe déjà, on compare les deux méthodes de rejet.

In [ ]:
sigmoid_path = reports_dir / "uncertainty_metrics.csv"

if sigmoid_path.exists():
    sigmoid_metrics = pd.read_csv(sigmoid_path)
    sigmoid_metrics.insert(0, "method", "sigmoid_confidence")
    feature_comparison = feature_metrics.copy()
    feature_comparison.insert(0, "method", "feature_distance")
    comparison = pd.concat([sigmoid_metrics, feature_comparison], ignore_index=True, sort=False)
    display(comparison[["method", "protocol", "threshold", "baseline_accuracy", "accepted_accuracy", "coverage", "uncertain_rate", "uncertain_error_rate"]])
else:
    print("Le fichier uncertainty_metrics.csv n'existe pas encore.")

## Résultats par product_type

In [ ]:
feature_by_product_type = pd.read_csv(reports_dir / "feature_distance_by_product_type.csv")
feature_by_product_type

In [ ]:
unseen_features = feature_by_product_type[feature_by_product_type["protocol"] == "unseen"]
ax = unseen_features.set_index("product_type")[["baseline_accuracy", "accepted_accuracy", "uncertain_rate"]].plot(
    kind="bar",
    figsize=(9, 5),
)
ax.set_ylim(0, 1)
ax.set_title("Feature-distance sur les catégories non vues")
ax.set_ylabel("Valeur")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Points à discuter

- La méthode rejette-t-elle davantage les catégories non vues ?
- Les images rejetées contiennent-elles beaucoup d'erreurs ?
- L'accuracy des images acceptées augmente-t-elle par rapport à la baseline ?
- La méthode est-elle meilleure que le seuil sur la confiance sigmoid ?

Cette analyse relie le projet aux notions de généralisation, de surapprentissage et de représentation interne vues en cours.